# Detecting Topic Merges and Splits in Dynamic Political Conversations

Cláudia Oliveira

Supervisor - Prof. Dr. Álvaro Figueira

Faculty of Science, University of Porto

In [1]:
%%capture
!pip install  sentence-transformers  gensim scikit-learn pandas  tqdm emoji rapidfuzz nltk
import nltk
nltk.download('stopwords')
!pip install -U numpy==1.26.4
!pip install -U scipy==1.11.4
!pip install -U hdbscan==0.8.33
!pip install -U bertopic==0.16.0

In [2]:
import pandas as pd
import numpy as np
import random
import ast
import torch
from datetime import timedelta
import matplotlib.pyplot as plt

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
from google.colab import files
uploaded = files.upload()

Saving 20news.csv to 20news.csv


### Topic Modeling for Static Datasets

In [ ]:
# ======================================================================
# 1. CONFIGURATION
# ======================================================================

min_docs_per_topic = 15
top_n_words = 10

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
from bertopic import BERTopic
import pandas as pd
import numpy as np
import ast

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

vectorizer_model = CountVectorizer(
    min_df=5,
    max_df=0.9
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric="euclidean"
)

# ======================================================================
# 2. LOAD DATA
# ======================================================================

df = pd.read_csv("20news.csv", encoding="utf-8", low_memory=False)

df["tokens"] = df["tokens"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df = df[df["tokens"].notnull()].reset_index(drop=True)

docs = [" ".join(tokens) for tokens in df["tokens"]]

# ======================================================================
# 3. COMPUTE DOCUMENT EMBEDDINGS
# ======================================================================

embeddings = embedding_model.encode(docs, show_progress_bar=True)

# ======================================================================
# 4. RUN BERTopic
# ======================================================================

topic_model = BERTopic(
    embedding_model=None,   
    umap_model=None,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=False
)

topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)

df["topic_id"] = topics

# ======================================================================
# 5. FILTER VALID TOPICS
# ======================================================================

topic_info = topic_model.get_topic_info()

valid_topics = topic_info[
    (topic_info["Count"] >= min_docs_per_topic) &
    (topic_info["Topic"] != -1)
]["Topic"].tolist()

# Topic words
topic_words = {
    tid: [w for w, _ in topic_model.get_topic(tid)[:top_n_words]]
    for tid in valid_topics
}

topic_doc_counts = {
    int(row["Topic"]): int(row["Count"])
    for _, row in topic_info.iterrows()
    if row["Topic"] in valid_topics
}

# ======================================================================
# 6. COMPUTE TOPIC EMBEDDINGS (REAL CENTROIDS)
# ======================================================================

topic_embeddings = {}

for tid in valid_topics:
    idx = df[df["topic_id"] == tid].index
    topic_embs = embeddings[idx]
    centroid = topic_embs.mean(axis=0)
    topic_embeddings[tid] = centroid

# ======================================================================
# 7. RESULTS
# ======================================================================

print("Number of valid topics:", len(valid_topics))

print("\nSample topics:")
for tid in valid_topics[:5]:
    print(f"\nTopic {tid}:")
    print("Words:", topic_words[tid])
    print("Docs:", topic_doc_counts[tid])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Number of valid topics: 24

Sample topics:

Topic 0:
Words: ['arab', 'jews', 'jewish', 'war', 'muslim', 'kill', 'peace', 'state', 'attack', 'occupy']
Docs: 614

Topic 1:
Words: ['turkish', 'soviet', 'genocide', 'russian', 'muslim', 'kill', 'massacre', 'road', 'woman', 'history']
Docs: 226

Topic 2:
Words: ['homosexual', 'cramer', 'clayton', 'gay', 'man', 'sexual', 'sex', 'male', 'number', 'partner']
Docs: 192

Topic 3:
Words: ['fire', 'fbi', 'gas', 'compound', 'tear', 'tank', 'koresh', 'child', 'survivor', 'inside']
Docs: 173

Topic 4:
Words: ['libertarian', 'power', 'party', 'system', 'constitution', 'problem', 'state', 'hendrick', 'steve', 'regulation']
Docs: 81


### Metrics for Topic Modeling

#### Function

In [5]:
def compute_tq(topic_words_dict, docs, dictionary, top_n=10):

    topic_words = [words[:top_n] for words in topic_words_dict.values() if words]
    if not topic_words:
        return None, None, None, None, None

    texts = [doc.split() for doc in docs]

    # ---- Coherence Models ----

    cm_cv = CoherenceModel(
        topics=topic_words,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )

    cm_npmi = CoherenceModel(
        topics=topic_words,
        texts=texts,
        dictionary=dictionary,
        coherence='c_npmi'
    )

    cm_umass = CoherenceModel(
        topics=topic_words,
        texts=texts,
        dictionary=dictionary,
        coherence='u_mass'
    )

    cv_scores = cm_cv.get_coherence_per_topic()
    npmi_scores = cm_npmi.get_coherence_per_topic()
    umass_scores = cm_umass.get_coherence_per_topic()

    # ---- Diversity ----

    K = len(topic_words)
    diversities = []
    for k, words_k in enumerate(topic_words):
        overlaps = 0
        for i, words_j in enumerate(topic_words):
            if i == k:
                continue
            overlaps += len(set(words_k) & set(words_j)) / len(words_k)
        redundancy = overlaps / (K - 1) if K > 1 else 0
        diversities.append(1 - redundancy)

    # ---- TQ ----
    tq = np.mean([c * d for c, d in zip(cv_scores, diversities)])

    return (
        tq,
        np.mean(cv_scores),
        np.mean(diversities),
        np.mean(npmi_scores),
        np.mean(umass_scores)
    )

#### Metrics

In [6]:
texts = [doc.split() for doc in docs]
dictionary = Dictionary(texts)

tq, cv_mean, diversity_mean, npmi_mean, umass_mean = compute_tq(
    topic_words,
    docs,
    dictionary,
    top_n=10
)

print("TQ:", tq)
print("CV:", cv_mean)
print("Diversity:", diversity_mean)
print("NPMI:", npmi_mean)
print("UMass:", umass_mean)

TQ: 0.653324377383394
CV: 0.6652777057685343
Diversity: 0.9818840579710146
NPMI: 0.09116210734423376
UMass: -1.9856592432874474


#### Purity

In [7]:
from sklearn.metrics import confusion_matrix

def compute_topic_purity(df, topic_column="topic_id", label_column="label"):

    df_valid = df[df[topic_column] != -1].copy()

    topics = df_valid[topic_column].astype(int).values
    labels = df_valid[label_column].astype(str).values

    unique_labels = sorted(df_valid[label_column].unique())
    label_to_id = {lbl: i for i, lbl in enumerate(unique_labels)}
    y_true = np.array([label_to_id[l] for l in labels])

    unique_topics = sorted(df_valid[topic_column].unique())
    topic_to_id = {t: i for i, t in enumerate(unique_topics)}
    y_pred = np.array([topic_to_id[t] for t in topics])

    cm = confusion_matrix(y_pred, y_true)

    max_intersections = cm.max(axis=1)

    purity = max_intersections.sum() / cm.sum()

    return purity, cm, unique_topics, unique_labels


purity, cm, unique_topics, unique_labels = compute_topic_purity(df)

print("\n======================================")
print(" TOPIC PURITY")
print("======================================")
print(f"Purity: {purity:.4f}")
print("Clusters:", len(unique_topics))
print("True labels:", len(unique_labels))

print("\nDominant category per topic:")
for i, topic_id in enumerate(unique_topics):
    dominant_label_index = cm[i].argmax()
    print(f"Topic {topic_id} → {unique_labels[dominant_label_index]} (count={cm[i].max()})")



 TOPIC PURITY
Purity: 0.9171
Clusters: 24
True labels: 3

Dominant category per topic:
Topic 0 → talk.politics.mideast (count=592)
Topic 1 → talk.politics.mideast (count=222)
Topic 2 → talk.politics.misc (count=187)
Topic 3 → talk.politics.guns (count=138)
Topic 4 → talk.politics.misc (count=65)
Topic 5 → talk.politics.guns (count=55)
Topic 6 → talk.politics.guns (count=39)
Topic 7 → talk.politics.guns (count=51)
Topic 8 → talk.politics.misc (count=29)
Topic 9 → talk.politics.misc (count=48)
Topic 10 → talk.politics.guns (count=41)
Topic 11 → talk.politics.guns (count=40)
Topic 12 → talk.politics.mideast (count=29)
Topic 13 → talk.politics.misc (count=33)
Topic 14 → talk.politics.misc (count=29)
Topic 15 → talk.politics.misc (count=28)
Topic 16 → talk.politics.misc (count=27)
Topic 17 → talk.politics.misc (count=26)
Topic 18 → talk.politics.guns (count=22)
Topic 19 → talk.politics.guns (count=17)
Topic 20 → talk.politics.guns (count=16)
Topic 21 → talk.politics.guns (count=17)
Topic 2